# ② 검색 — 색인 만들기 · 검색
담당: 검색 · 코드: `rag/retrieval/indexing.py`, `rag/retrieval/retrieval.py`

01 전처리의 `artifacts/chunks.jsonl` 로 Dense(KURE-v1 임베딩 → FAISS)와 BM25(Kiwi 형태소) 색인을 만들어 `artifacts/index/` 에 저장합니다.
담당자가 제공한 `data/data_list.csv`(UTF-8, `파일명` · `발주 기관` 열)가 필요합니다. 없으면 색인 만들기 전에 요청하세요. 사용자 사전·별칭·부정어·요구사항 코드 토큰이 바뀌었으므로 기존 색인도 다시 만드세요.
검색 전에 OpenAI API로 질문을 재작성합니다(유료). 재작성한 질문으로 Dense·BM25 검색을 하고, 원본 질문에 나온 발주기관 문서 안에서만 찾습니다.
`.env` 에 Langfuse 키가 있으면 검색할 때마다 Langfuse Cloud 에 `retrieve-context` 로 기록됩니다 (없어도 똑같이 동작).

In [ ]:
import pandas as pd

from rag.config import ARTIFACTS, load_config, read_jsonl, write_jsonl
from rag.retrieval.indexing import build_index
from rag.retrieval.retrieval import HybridIndex
from rag.retrieval.rewrite import rewrite_question
from rag.usage import usage_summary

cfg = load_config()

## 설정
초기값은 `config.yaml`에서 읽습니다. 아래 변수의 오른쪽을 원하는 값으로 바꾸고 설정 셀부터 다시 실행하세요.
`DENSE_WEIGHT`는 0~1이며 BM25 비율은 `1 - DENSE_WEIGHT`입니다. 0은 BM25만, 1은 Dense만 사용합니다.
적용한 rewrite·검색 설정을 `artifacts/retrieval_settings.jsonl`에 저장합니다. 04는 이 파일을 읽어 같은 설정으로 검색하므로, 값을 바꾼 뒤에는 04도 다시 실행하세요.


In [ ]:
REWRITE_MODEL = cfg["rewrite"]["model"]                     # 예: "gpt-5-mini"
REWRITE_REASONING_EFFORT = cfg["rewrite"]["reasoning_effort"]  # 예: "low"
REWRITE_MAX_TOKENS = cfg["rewrite"]["max_tokens"]             # 출력·추론 토큰 한도
REWRITE_PRICES = cfg["rewrite"]["usd_per_1m_tokens"]          # 입력 · 캐시 입력 · 출력 단가
DENSE_WEIGHT = cfg["retrieval"]["dense_weight"]               # 예: 0.6 (Dense 60%, BM25 40%)

cfg["rewrite"].update(model=REWRITE_MODEL, reasoning_effort=REWRITE_REASONING_EFFORT,
                      max_tokens=REWRITE_MAX_TOKENS, usd_per_1m_tokens=REWRITE_PRICES)
cfg["retrieval"]["dense_weight"] = DENSE_WEIGHT
write_jsonl([{"rewrite": cfg["rewrite"], "retrieval": cfg["retrieval"]}], ARTIFACTS / "retrieval_settings.jsonl")
cfg["rewrite"], {"Dense": DENSE_WEIGHT, "BM25": 1 - DENSE_WEIGHT}


## 1. 색인 만들기

In [ ]:
chunks = read_jsonl(ARTIFACTS / "chunks.jsonl")
build_index(chunks, cfg, ARTIFACTS / "index")

## 2. 검색해 보기
위 설정 셀에서 rewrite·검색 비율을 조절한 결과를 확인하세요. token_budget(기본 2500)·candidate_k의 값은 `config.yaml`에서 읽습니다. 요구사항 코드 일치 청크를 먼저 담고, 거의 같은 본문은 제외합니다.

In [ ]:
index = HybridIndex(ARTIFACTS, cfg)
QUESTION = "고려대학교 차세대 포털 시스템 사업의 응답 시간 요구사항은?"
rewrite = rewrite_question(QUESTION, cfg["rewrite"])
print(rewrite)
pd.DataFrame(index.search(QUESTION, query=rewrite["question"]))[["score", "title", "page", "kind", "section", "body"]]

## API 사용량
02와 04의 rewrite 호출은 모두 `artifacts/api_usage.csv`에 자동 추가됩니다. 아래 표는 누적 사용량입니다.

In [ ]:
usage_summary("rewrite")